# Phase 4B-2 — External Validation: Ground Truth Tests
Tests: Dice WT/TC/ET | Volume Rank Correlation | Trajectory Spearman | Classification Accuracy


In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "monai"], capture_output=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "nibabel", "einops", "scikit-image", "scipy"], capture_output=True)
import torch
print(f"Ready | {torch.__version__} | {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")


In [ ]:
import os, json, warnings, random
import numpy as np
import pandas as pd
import torch, torch.nn.functional as F
import nibabel as nib
from pathlib import Path
from scipy.stats import spearmanr
warnings.filterwarnings("ignore")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type == "cuda":
    try:
        _ = torch.zeros(4,4,device="cuda") @ torch.zeros(4,4,device="cuda")
        print(f"Device: cuda OK")
    except:
        device = torch.device("cpu")
        print(f"Device: CPU (CUDA fallback)")
else:
    print(f"Device: CPU")

PATCH = (128,128,128)
OUTPUT_ROOT = Path("/kaggle/working/mu_glioma_validation")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)


In [ ]:
import monai.transforms as T
from monai.transforms import MapTransform

# --- Symlinks ---
SYMLINK_DIR = Path('/kaggle/working/nifti_links')
n_links = 0
for nii_gz in Path('/kaggle/input').rglob('*.nii_gz'):
    real_name = nii_gz.name.replace('.nii_gz', '.nii.gz')
    for pi, part in enumerate(nii_gz.parts):
        if part.startswith("PatientID"):
            link = SYMLINK_DIR / Path(*nii_gz.parts[pi:-1]) / real_name
            link.parent.mkdir(parents=True, exist_ok=True)
            if not link.exists():
                os.symlink(str(nii_gz), str(link))
                n_links += 1
            break
print(f"Symlinks: {n_links}")

# --- MU root ---
MU_ROOT = None
for sr in [SYMLINK_DIR, Path("/kaggle/input")]:
    if not sr.exists(): continue
    for c in sr.rglob("PatientID_0003"):
        if c.is_dir(): MU_ROOT = c.parent; break
    if MU_ROOT: break
print(f"MU-Glioma root: {MU_ROOT}")

# --- Load inference results ---
results_df = pd.read_csv(next(Path("/kaggle/input").rglob("mu_glioma_results.csv")))
vol_df     = pd.read_csv(next(Path("/kaggle/input").rglob("mu_glioma_volumes.csv")))
print(f"Inference results: {len(results_df)} patients | Volumes: {len(vol_df)} scans")


In [ ]:
# =======================================================
# TEST 1: DICE SCORES — SwinUNETR prediction vs GT masks
# =======================================================
from monai.networks.nets import SwinUNETR
from monai.data import Dataset, DataLoader

class ConvertBrats(MapTransform):
    # Labels: 1=NETC, 2=SNFH, 3=ET, 4=RC(excluded) -> [WT, TC, ET]
    def __call__(self, data):
        d = dict(data)
        for key in self.key_iterator(d):
            img = d[key]
            if img.ndim == 4 and img.shape[0] == 1: img = img.squeeze(0)
            d[key] = (np.stack([(img==1)|(img==2)|(img==3),
                                (img==1)|(img==3),
                                img==3], 0).astype(np.float32))
        return d

val_tfm = T.Compose([
    T.LoadImaged(keys=['image','label']),
    T.EnsureChannelFirstd(keys=['image','label']),
    T.Orientationd(keys=['image','label'], axcodes='RAS'),
    T.CropForegroundd(keys=['image','label'], source_key='image', allow_smaller=True),
    T.NormalizeIntensityd(keys='image', nonzero=True, channel_wise=True),
    ConvertBrats(keys=['label']),
    T.EnsureTyped(keys=['image','label'], dtype=torch.float32),
])

ckpt_path = next(Path("/kaggle/input").rglob("swinunetr_best.pth"))
model = SwinUNETR(in_channels=4, out_channels=3, feature_size=48, use_checkpoint=False).to(device)
ckpt  = torch.load(ckpt_path, map_location=device, weights_only=False)
state = ckpt.get("model_state_dict", ckpt.get("state_dict", ckpt.get("model", ckpt)))
state = {k.replace("module.",""):v for k,v in state.items() if not k.startswith("emb_head")}
missing, unexpected = model.load_state_dict(state, strict=False)
model.eval()

# Diagnostic: check how many parameters actually loaded
n_loaded = sum(p.numel() for n,p in model.named_parameters()
               if n not in missing and not any(u in n for u in unexpected))
n_total  = sum(p.numel() for p in model.parameters())
pct = 100 * n_loaded / n_total if n_total > 0 else 0
print(f"SwinUNETR loaded | params loaded: {n_loaded:,}/{n_total:,} ({pct:.1f}%)")
if missing: print(f"  Missing keys (first 5): {missing[:5]}")
print(f"  Checkpoint keys (first 5): {list(state.keys())[:5]}")

patients = sorted([d for d in MU_ROOT.iterdir() if d.is_dir() and d.name.startswith("PatientID")])
random.seed(42)
sample = random.sample(patients, min(60, len(patients)))

val_dicts = []
for pdir in sample:
    for tp_dir in sorted(pdir.iterdir()):
        if not tp_dir.name.startswith("Timepoint"): continue
        pid = pdir.name; tp = tp_dir.name
        files = {m: tp_dir/f"{pid}_{tp}_brain_{m}.nii.gz"
                 for m in ["t1n","t1c","t2w","t2f"]}
        mask  = tp_dir/f"{pid}_{tp}_tumorMask.nii.gz"
        if all(f.exists() for f in list(files.values())+[mask]):
            val_dicts.append({"image":[str(files["t1n"]),str(files["t1c"]),
                                        str(files["t2w"]),str(files["t2f"])],
                              "label":str(mask), "patient_id":pid})
            break
print(f"Dice subset: {len(val_dicts)} scans")

def dice(pred, gt):
    inter = (pred & gt).sum().float()
    denom = pred.sum() + gt.sum()
    return float('nan') if denom==0 else (2*inter/denom).item()

dice_wt, dice_tc, dice_et = [], [], []
pred_wt_vols, pred_tc_vols, pred_et_vols = [], [], []
gt_wt_vols, gt_tc_vols, gt_et_vols = [], [], []
n_skip = 0
ds = Dataset(val_dicts, val_tfm)

def safe_iter(loader):
    it = iter(loader); ns = 0
    while True:
        try: yield next(it)
        except StopIteration:
            if ns: print(f"  Loader skipped {ns}")
            return
        except Exception as e:
            if any(k in str(e) for k in ['gzip','corrupt','ImageFile','LoadImaged','applying']):
                ns+=1; continue
            raise

with torch.no_grad():
    for idx, batch in enumerate(safe_iter(DataLoader(ds, batch_size=1, shuffle=False, num_workers=0))):
        try:
            img = F.interpolate(batch['image'].to(device), list(PATCH), mode='trilinear', align_corners=False)
            lbl = F.interpolate(batch['label'].to(device), list(PATCH), mode='nearest')
            _logits = model(img)
            if idx == 0:
                print(f"  [Diag] logit range: {_logits.min().item():.2f} to {_logits.max().item():.2f}")
                print(f"  [Diag] sigmoid>0.5: WT={(_logits[0,0]>0).sum().item()} TC={(_logits[0,1]>0).sum().item()} ET={(_logits[0,2]>0).sum().item()}")
                print(f"  [Diag] GT WT voxels: {lbl[0,0].sum().item():.0f}")
            pred = (torch.sigmoid(_logits) > 0.5)[0]
            gt   = lbl[0].bool()
            dw, dt, de = dice(pred[0], gt[0]), dice(pred[1], gt[1]), dice(pred[2], gt[2])
            if not any(np.isnan([dw,dt,de])):
                dice_wt.append(dw); dice_tc.append(dt); dice_et.append(de)
                # Save predicted + GT voxel counts for TEST 2
                pred_wt_vols.append(int(pred[0].sum().item()))
                pred_tc_vols.append(int(pred[1].sum().item()))
                pred_et_vols.append(int(pred[2].sum().item()))
                gt_wt_vols.append(int(gt[0].sum().item()))
                gt_tc_vols.append(int(gt[1].sum().item()))
                gt_et_vols.append(int(gt[2].sum().item()))
            if idx==0:
                print(f"  [Sample] {batch['patient_id'][0]}: WT={dw:.3f} TC={dt:.3f} ET={de:.3f}")
            del img, lbl
            if device.type=='cuda': torch.cuda.empty_cache()
        except Exception: n_skip+=1

print(f"\n{'='*50}")
print(f"  TEST 1: DICE  (n={len(dice_wt)} scans, skipped={n_skip})")
print(f"  WT: {np.mean(dice_wt):.3f} +/- {np.std(dice_wt):.3f}")
print(f"  TC: {np.mean(dice_tc):.3f} +/- {np.std(dice_tc):.3f}")
print(f"  ET: {np.mean(dice_et):.3f} +/- {np.std(dice_et):.3f}")
for name,arr,thr in [('WT',dice_wt,0.75),('TC',dice_tc,0.65),('ET',dice_et,0.55)]:
    mu=np.mean(arr)
    print(f"  {name}: {mu:.3f} vs >={thr} -> {'PASS' if mu>=thr else 'BELOW THRESHOLD'}")


In [ ]:
# =======================================================
# TEST 2: VOLUME RANK CORRELATION (extracted 128^3 vs GT mm^3)
# TEST 3: TRAJECTORY SPEARMAN   (predicted ratio vs GT ratio)
# TEST 4: CLASSIFICATION ACCURACY
# =======================================================
print("Computing GT volumes from masks...")
gt_rows = []
for _, row in vol_df.iterrows():
    pid, tp = str(row['patient_id']), int(row['timepoint'])
    mask_f = MU_ROOT / pid / f"Timepoint_{tp}" / f"{pid}_Timepoint_{tp}_tumorMask.nii.gz"
    if not mask_f.exists():
        gt_rows.append({'patient_id':pid,'timepoint':tp,'gt_wt':np.nan,'gt_tc':np.nan,'gt_et':np.nan})
        continue
    m = nib.load(str(mask_f)).get_fdata().astype(np.int32)
    gt_rows.append({'patient_id':pid,'timepoint':tp,
                    'gt_wt':int(((m==1)|(m==2)|(m==3)).sum()),
                    'gt_tc':int(((m==1)|(m==3)).sum()),
                    'gt_et':int((m==3).sum())})

gt_df  = pd.DataFrame(gt_rows)
merged = vol_df.merge(gt_df, on=['patient_id','timepoint']).dropna(subset=['gt_wt'])
print(f"Matched scans: {len(merged)}/{len(vol_df)}")

# TEST 2: SwinUNETR PREDICTED volumes vs GT (from Dice loop)
if pred_wt_vols:
    rho_wt,p_wt = spearmanr(pred_wt_vols, gt_wt_vols)
    rho_tc,p_tc = spearmanr(pred_tc_vols, gt_tc_vols)
    rho_et,p_et = spearmanr(pred_et_vols, gt_et_vols)
    n_v = len(pred_wt_vols)
else:
    rho_wt=rho_tc=rho_et=float('nan'); p_wt=p_tc=p_et=1.0; n_v=0
print(f"\n{'='*50}")
print(f"  TEST 2: PREDICTED vs GT VOLUME RANK (n={n_v})")
print(f"  WT: rho={rho_wt:.3f} p={p_wt:.2e} -> {'PASS' if rho_wt>=0.70 else 'BELOW'}")
print(f"  TC: rho={rho_tc:.3f} p={p_tc:.2e} -> {'PASS' if rho_tc>=0.60 else 'BELOW'}")
print(f"  ET: rho={rho_et:.3f} p={p_et:.2e} -> {'PASS' if rho_et>=0.50 else 'BELOW'}")

# --- Trajectory ---
traj_rows = []
for pid in results_df['patient_id'].unique():
    pid_m = merged[merged['patient_id']==pid].sort_values('timepoint')
    if len(pid_m)<2: continue
    v0 = float(pid_m.iloc[0]['gt_wt'])
    v1 = float(pid_m.iloc[-1]['gt_wt'])
    if v0 < 1: continue
    gt_r = (v1-v0)/v0
    pr = results_df[results_df['patient_id']==pid]['predicted_ratio']
    if pr.empty: continue
    traj_rows.append({'patient_id':pid,'gt_ratio':gt_r,'pred_ratio':float(pr.iloc[0])})

traj_df = pd.DataFrame(traj_rows)
print(f"\n  TEST 3: TRAJECTORY SPEARMAN  (n={len(traj_df)} patients)")
if len(traj_df) >= 10:
    rho_t,p_t = spearmanr(traj_df['pred_ratio'], traj_df['gt_ratio'])
    mae = float(np.abs(traj_df['pred_ratio']-traj_df['gt_ratio']).mean())
    print(f"  Spearman rho={rho_t:.3f} p={p_t:.2e} -> {'PASS' if rho_t>=0.3 and p_t<0.05 else 'BELOW'}")
    print(f"  MAE: {mae:.3f}")

    def cls_gt(r): return 'progressive' if r>=0.25 else ('responder' if r<=-0.25 else 'stable')
    traj_df['gt_class'] = traj_df['gt_ratio'].apply(cls_gt)
    traj_df = traj_df.merge(results_df[['patient_id','trajectory_class']], on='patient_id')
    traj_df = traj_df.rename(columns={'trajectory_class':'pred_class_orig'})
    def cls_abs(r): return 'progressive' if r>=0.25 else ('responder' if r<=-0.25 else 'stable')
    traj_df['pred_class_abs'] = traj_df['pred_ratio'].apply(cls_abs)
    p_prog = np.percentile(traj_df['pred_ratio'], 51)
    p_resp = np.percentile(traj_df['pred_ratio'], 30)
    def cls_pct(r):
        if r >= p_prog: return 'progressive'
        if r <= p_resp: return 'responder'
        return 'stable'
    traj_df['pred_class_pct'] = traj_df['pred_ratio'].apply(cls_pct)
    acc_abs = (traj_df['gt_class']==traj_df['pred_class_abs']).mean()
    acc_pct = (traj_df['gt_class']==traj_df['pred_class_pct']).mean()
    acc = acc_pct
    print(f"\n  TEST 4: CLASSIFICATION (n={len(traj_df)})")
    print(f"  GT: {traj_df['gt_class'].value_counts().to_dict()}")
    print(f"  Absolute +-25%: acc={acc_abs:.3f}  {traj_df['pred_class_abs'].value_counts().to_dict()}")
    print(f"  Percentile p30/p51: acc={acc_pct:.3f}  {traj_df['pred_class_pct'].value_counts().to_dict()}")
    print(f"  -> Abs: {'PASS' if acc_abs>=0.5 else 'BELOW'} | Pct: {'PASS' if acc_pct>=0.5 else 'BELOW'}")

traj_df.to_csv(OUTPUT_ROOT/"validation_trajectory.csv", index=False)
gt_df.to_csv(OUTPUT_ROOT/"gt_volumes.csv", index=False)
print(f"\nSaved to {OUTPUT_ROOT}")


In [ ]:
print("\n" + "="*60)
print("  MU-GLIOMA-POST — FINAL VALIDATION REPORT")
print("="*60)
print(f"\n  Segmentation (Dice, n={len(dice_wt)})")
print(f"    WT: {np.mean(dice_wt):.3f}  TC: {np.mean(dice_tc):.3f}  ET: {np.mean(dice_et):.3f}")
print(f"\n  Volume ranking (Spearman, n={len(merged)})")
print(f"    WT rho={rho_wt:.3f}  TC rho={rho_tc:.3f}  ET rho={rho_et:.3f}")
if len(traj_df) >= 10:
    print(f"\n  Trajectory (n={len(traj_df)})")
    print(f"    Spearman rho={rho_t:.3f}  MAE={mae:.3f}")
    print(f"    Classification accuracy={acc:.3f}")
print("="*60)
